# 05 — Pretraining Exploration

Explores the pretraining pipeline — tokenized dataset statistics, training dynamics, loss curves, learning rate schedule, and checkpoint analysis.

Run after `make tokenize` and `make pretrain`.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import torch

DATA_DIR = Path('../data')
RESULTS_DIR = Path('../results')
TOKENIZED_DIR = DATA_DIR / 'tokenized'

## 1. Tokenized dataset statistics

In [ ]:
meta_path = TOKENIZED_DIR / 'train.json'
bin_path = TOKENIZED_DIR / 'train.bin'

if meta_path.exists():
    with open(meta_path) as f:
        meta = json.load(f)

    print(f"Token file:         {bin_path}")
    print(f"Total tokens:       {meta['n_tokens']:,} ({meta['n_tokens'] / 1e9:.2f}B)")
    print(f"Total documents:    {meta['n_docs']:,}")
    print(f"Avg tokens/doc:     {meta['n_tokens'] // meta['n_docs']:,}")
    print(f"File size:          {bin_path.stat().st_size / 1e9:.2f} GB")
    print(f"Dtype:              {meta['dtype']}")
    print(f"EOS token ID:       {meta['eos_id']}")
else:
    print("Tokenized dataset not found — run: make tokenize")

## 2. Dataset examples

In [ ]:
if bin_path.exists():
    from pretrain.data.dataset import PretrainingDataset
    from tokenizers import Tokenizer

    dataset = PretrainingDataset(bin_path, seq_len=2048)
    tokenizer = Tokenizer.from_file(str(DATA_DIR / 'tokenizer' / 'slm_tokenizer.json'))

    print(f"Dataset examples: {len(dataset):,}")
    print(f"Sequence length:  {dataset.seq_len}")
    print()

    # Show a sample example
    item = dataset[0]
    print(f"input_ids shape: {item['input_ids'].shape}")
    print(f"labels shape:    {item['labels'].shape}")
    print(f"First 10 input IDs: {item['input_ids'][:10].tolist()}")
    print(f"First 10 labels:    {item['labels'][:10].tolist()}")
    print()

    # Decode first 200 tokens
    decoded = tokenizer.decode(item['input_ids'][:200].tolist())
    print(f"Decoded (first 200 tokens):")
    print(decoded[:500])

## 3. Token distribution

In [ ]:
if bin_path.exists():
    # Sample 1M tokens for analysis
    data = np.memmap(str(bin_path), dtype=np.uint16, mode='r')
    sample = data[:1_000_000]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Token ID distribution
    axes[0].hist(sample, bins=100, color='#4C72B0', alpha=0.85, edgecolor='white')
    axes[0].set_xlabel('Token ID')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Token ID Distribution (1M sample)', fontsize=11, fontweight='bold')
    axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

    # Top 50 most frequent tokens
    from collections import Counter
    counter = Counter(sample.tolist())
    top50 = counter.most_common(50)
    ids, freqs = zip(*top50)
    axes[1].bar(range(50), freqs, color='#DD8452', alpha=0.85)
    axes[1].set_xlabel('Rank')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Top 50 Most Frequent Tokens', fontsize=11, fontweight='bold')
    axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

    plt.tight_layout()
    plt.show()

    # EOS token frequency
    eos_count = (sample == meta['eos_id']).sum()
    print(f"EOS tokens in 1M sample: {eos_count:,} ({100*eos_count/1e6:.2f}%)")
    print(f"Avg tokens between EOS:  {1_000_000 // max(eos_count, 1):,}")

## 4. LR schedule visualization

In [ ]:
# Visualize cosine LR schedule for each model size
configs = [
    {'name': 'slm-125m', 'max_steps': 150_000, 'warmup': 2000, 'lr': 3e-4, 'color': '#4C72B0'},
    {'name': 'slm-350m', 'max_steps': 500_000, 'warmup': 2000, 'lr': 2e-4, 'color': '#DD8452'},
    {'name': 'slm-1b',   'max_steps': 1_500_000, 'warmup': 2000, 'lr': 1e-4, 'color': '#55A868'},
]

def cosine_lr(step, max_steps, warmup_steps, max_lr, min_lr_ratio=0.1):
    min_lr = max_lr * min_lr_ratio
    if step < warmup_steps:
        return max_lr * step / warmup_steps
    progress = (step - warmup_steps) / (max_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))

fig, ax = plt.subplots(figsize=(12, 4))

for cfg in configs:
    steps = np.linspace(0, cfg['max_steps'], 1000, dtype=int)
    lrs = [cosine_lr(s, cfg['max_steps'], cfg['warmup'], cfg['lr']) for s in steps]
    ax.plot(steps / 1000, lrs, label=cfg['name'], color=cfg['color'], linewidth=2)

ax.set_xlabel('Steps (thousands)')
ax.set_ylabel('Learning Rate')
ax.set_title('Cosine LR Schedule — All Model Sizes', fontsize=12, fontweight='bold')
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0e}'))
plt.tight_layout()
plt.show()

## 5. Training curves (if checkpoint exists)

In [ ]:
# Load trainer_state.json from latest checkpoint
model_dir = RESULTS_DIR / 'slm-125m'
state_files = list(model_dir.glob('**/trainer_state.json'))

if state_files:
    state_file = sorted(state_files)[-1]
    with open(state_file) as f:
        state = json.load(f)

    log_history = state.get('log_history', [])
    train_logs = [x for x in log_history if 'loss' in x and 'eval_loss' not in x]
    eval_logs = [x for x in log_history if 'eval_loss' in x]

    print(f"Loaded training state from: {state_file}")
    print(f"Train log entries: {len(train_logs)}")
    print(f"Eval log entries:  {len(eval_logs)}")
    print(f"Best eval loss:    {state.get('best_metric', 'N/A')}")
else:
    print("No training checkpoints found — run: make pretrain")
    train_logs = []
    eval_logs = []

In [ ]:
if train_logs:
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))

    train_steps = [x['step'] for x in train_logs]
    train_loss = [x['loss'] for x in train_logs]
    train_lr = [x.get('learning_rate', 0) for x in train_logs]
    train_grad = [x.get('grad_norm', 0) for x in train_logs]

    eval_steps = [x['step'] for x in eval_logs]
    eval_loss = [x['eval_loss'] for x in eval_logs]

    # Training loss
    axes[0, 0].plot(train_steps, train_loss, color='#4C72B0', alpha=0.6, linewidth=0.8, label='train loss')
    # Smoothed
    if len(train_loss) > 20:
        smoothed = np.convolve(train_loss, np.ones(20)/20, mode='valid')
        axes[0, 0].plot(train_steps[10:-9], smoothed, color='#4C72B0', linewidth=2, label='smoothed')
    if eval_logs:
        axes[0, 0].plot(eval_steps, eval_loss, color='#DD8452', linewidth=2, marker='o', markersize=3, label='eval loss')
    axes[0, 0].set_xlabel('Step')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Training & Eval Loss', fontsize=11, fontweight='bold')
    axes[0, 0].legend(fontsize=9)

    # Perplexity
    train_ppl = [math.exp(min(l, 10)) for l in train_loss]
    axes[0, 1].plot(train_steps, train_ppl, color='#55A868', alpha=0.6, linewidth=0.8)
    if eval_logs:
        eval_ppl = [math.exp(min(l, 10)) for l in eval_loss]
        axes[0, 1].plot(eval_steps, eval_ppl, color='#C44E52', linewidth=2, marker='o', markersize=3, label='eval')
    axes[0, 1].set_xlabel('Step')
    axes[0, 1].set_ylabel('Perplexity')
    axes[0, 1].set_title('Perplexity', fontsize=11, fontweight='bold')

    # Learning rate
    axes[1, 0].plot(train_steps, train_lr, color='#8172B2', linewidth=1.5)
    axes[1, 0].set_xlabel('Step')
    axes[1, 0].set_ylabel('Learning Rate')
    axes[1, 0].set_title('Learning Rate Schedule', fontsize=11, fontweight='bold')
    axes[1, 0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0e}'))

    # Gradient norm
    axes[1, 1].plot(train_steps, train_grad, color='#C44E52', alpha=0.6, linewidth=0.8)
    axes[1, 1].axhline(1.0, color='black', linestyle='--', alpha=0.4, label='clip threshold')
    axes[1, 1].set_xlabel('Step')
    axes[1, 1].set_ylabel('Gradient Norm')
    axes[1, 1].set_title('Gradient Norm', fontsize=11, fontweight='bold')
    axes[1, 1].legend(fontsize=9)

    fig.suptitle('slm-125m Pretraining Dynamics', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 6. Chinchilla analysis

In [ ]:
# How far through Chinchilla optimal are we?
if bin_path.exists() and train_logs:
    current_step = train_logs[-1]['step']
    tokens_per_step = 32 * 2048  # global_batch * seq_len
    tokens_trained = current_step * tokens_per_step
    chinchilla_optimal = 125e6 * 20  # 20x params

    print(f"Current step:          {current_step:,}")
    print(f"Tokens trained so far: {tokens_trained / 1e9:.2f}B")
    print(f"Chinchilla optimal:    {chinchilla_optimal / 1e9:.2f}B tokens")
    print(f"Progress:              {100 * tokens_trained / chinchilla_optimal:.1f}%")
    print(f"Remaining:             {(chinchilla_optimal - tokens_trained) / 1e9:.2f}B tokens")

## 7. Model checkpoint inspection

In [ ]:
final_dir = RESULTS_DIR / 'slm-125m' / 'final'

if final_dir.exists():
    from transformers import AutoModelForCausalLM, AutoConfig
    from model import SLMConfig, SLMForCausalLM
    from transformers import AutoConfig
    AutoConfig.register('slm', SLMConfig)

    config = SLMConfig.from_pretrained(str(final_dir))
    print("Model config:")
    print(f"  num_hidden_layers:    {config.num_hidden_layers}")
    print(f"  hidden_size:          {config.hidden_size}")
    print(f"  num_attention_heads:  {config.num_attention_heads}")
    print(f"  num_key_value_heads:  {config.num_key_value_heads}")
    print(f"  max_position_emb:     {config.max_position_embeddings}")
    print(f"  vocab_size:           {config.vocab_size}")
    print()

    # Load model weights (CPU)
    model = SLMForCausalLM.from_pretrained(str(final_dir), device_map='cpu')
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Parameters: {n_params:,} ({n_params / 1e6:.1f}M)")

    # Quick forward pass
    input_ids = torch.randint(0, config.vocab_size, (1, 32))
    with torch.no_grad():
        out = model(input_ids)
    print(f"Output logits shape: {out.logits.shape}")
    print(f"Output loss: {out.loss}")
else:
    print(f"Final checkpoint not found at {final_dir}")
    print("Run: make pretrain")

## 8. Quick generation test

In [ ]:
if final_dir.exists():
    from tokenizers import Tokenizer

    tokenizer = Tokenizer.from_file(str(DATA_DIR / 'tokenizer' / 'slm_tokenizer.json'))

    prompts = [
        "The history of artificial intelligence",
        "def quicksort(arr):",
        "The capital of France is",
    ]

    model.eval()
    for prompt in prompts:
        input_ids = torch.tensor([tokenizer.encode(prompt).ids])
        with torch.no_grad():
            output = model.generate(
                input_ids,
                max_new_tokens=50,
                temperature=0.8,
                top_p=0.95,
                do_sample=True,
                pad_token_id=0,
                eos_token_id=3,
            )
        generated = tokenizer.decode(output[0].tolist(), skip_special_tokens=True)
        print(f"Prompt: {prompt}")
        print(f"Output: {generated}")
        print()